# Testing Predict For Meal

In [23]:
import pickle
import pandas as pd
import numpy as np
import os

In [24]:
# ==========================================
# 1. LOAD MODEL (Robust Path Handling)
# ==========================================

# Define path (Update this to your actual path if needed)
file_path = '../../models/model_meal.pickle' 
# Or use the absolute path you provided:
# file_path = r'C:\Users\vince\Documents\PPTI\CAWU_4_VincentF\VIKTORIA\project\viktorifit-ml\models\model_meal.pickle'

with open(file_path, 'rb') as f:
    meal_data = pickle.load(f)

knn = meal_data['knn_model']
scaler = meal_data['scaler']
db_meal = meal_data['meal_db']
features_model = meal_data['features'] # These are the columns the model expects

print("✅ Model Meal Loaded Successfully.")
print(f"   Features Expected: {features_model}")

# ==========================================
# 2. GENERATE SCHEDULE FUNCTION
# ==========================================

def generate_full_day_plan(target_harian, freq_makan):
    
    print("\n" + "="*50)
    print("🥗 GENERATING MEAL PLAN")
    print("="*50)
    
    # 1. Calculate Target Per Meal (Average)
    # We map your input keys to standard keys for easier math
    avg_cal = target_harian['Daily_Calories'] / freq_makan
    avg_prot = target_harian['Target_Protein_g'] / freq_makan
    avg_carbs = target_harian['Target_Carbs_g'] / freq_makan
    avg_fat = target_harian['Target_Fat_g'] / freq_makan

    print(f"🎯 DAILY TARGET : {target_harian['Daily_Calories']} kcal")
    print(f"🍽️ FREQUENCY    : {freq_makan}x meals")
    print(f"🎯 PER MEAL     : ~{avg_cal:.0f} kcal | {avg_prot:.1f}g Prot")
    print("-" * 50)
    
    # 2. Prepare Input Data for AI
    # [IMPORTANT Fix]: We must map values to the EXACT feature names the model knows.
    # We create a dictionary first, then convert to DataFrame using the model's feature list.
    
    # Mapping your 'user_target' keys to the 'features_model' columns
    # Assuming features_model are ['Energy', 'Protein', 'Fat', 'Carbs'] or similar.
    # We use a safe mapping approach:
    
    input_data_map = {
        'Energy': avg_cal,
        'Protein': avg_prot,
        'Carbs': avg_carbs,
        'Fat': avg_fat,
    }
    
    # Create DataFrame with the EXACT columns the scaler expects
    try:
        input_df = pd.DataFrame([input_data_map])[features_model]
    except KeyError as e:
        print(f"❌ ERROR: Column Name Mismatch. Your model expects {features_model}")
        print(f"   But we calculated these keys: {list(input_data_map.keys())}")
        return

    # Scale the input
    input_scaled = scaler.transform(input_df)
    
    # 3. SEARCH LOOP
    menu_sudah_dipilih = []
    total_daily_cal = 0
    total_daily_prot = 0
    
    for i in range(1, freq_makan + 1):
        
        # Search for candidates (Get top 20 to ensure variety)
        distances, indices = knn.kneighbors(input_scaled, n_neighbors=10)
        
        # Get candidate rows
        candidates = db_meal.iloc[indices[0]].copy()
        
        # [FILTER] Remove duplicates (foods already eaten today)
        candidates = candidates[~candidates['Food Items'].isin(menu_sudah_dipilih)]
        
        # Safety: If filtering removed everyone, reload the original candidates
        if candidates.empty:
            candidates = db_meal.iloc[indices[0]].copy()
        
        # Pick the Top 1 Match
        pilihan = candidates.iloc[0]
        
        # [SMART PORTIONING]
        # Instead of 1 portion, we adjust portion to match calorie target exactly
        # Logic: Target 500kcal, Food is 250kcal -> Portion = 2.0
        porsi = avg_cal / pilihan['Energy'] # Change 'Energy' to 'Energy kcal' if needed
        
        # Round portion to nearest 0.5 (e.g., 1.0, 1.5, 2.0) for realism
        porsi = round(porsi * 2) / 2
        if porsi < 0.5: porsi = 0.5
        if porsi > 3.0: porsi = 3.0 # Cap max portion to avoid absurdity
        
        # Record Selection
        menu_sudah_dipilih.append(pilihan['Food Items'])
        
        # Calculate Intake
        in_cal = pilihan['Energy'] * porsi
        in_prot = pilihan['Protein'] * porsi
        
        total_daily_cal += in_cal
        total_daily_prot += in_prot
        
        # Print Result
        print(f"⏰ MEAL #{i}")
        print(f"   Menu    : {pilihan['Food Items']}")
        print(f"   Portion : {porsi} x Serving ({pilihan['Energy']:.0f} kcal/srv)")
        print(f"   Total   : {in_cal:.0f} kcal | {in_prot:.1f}g Prot")
        print("-" * 30)

    # 4. FINAL SUMMARY
    print("="*50)
    print("📊 DAILY SUMMARY")
    print(f"   Calories : {total_daily_cal:.0f} / {target_harian['Daily_Calories']} ({total_daily_cal/target_harian['Daily_Calories']:.0%})")
    print(f"   Protein  : {total_daily_prot:.0f}g / {target_harian['Target_Protein_g']}g")
    print("="*50)

# ==========================================
# 3. TEST RUN
# ==========================================

user_target = {
    'Daily_Calories': 2200,
    'Target_Protein_g': 180,
    'Target_Carbs_g': 200,
    'Target_Fat_g': 70
}

generate_full_day_plan(user_target, freq_makan=3)

✅ Model Meal Loaded Successfully.
   Features Expected: ['Energy', 'Protein', 'Fat', 'Carbs']

🥗 GENERATING MEAL PLAN
🎯 DAILY TARGET : 2200 kcal
🍽️ FREQUENCY    : 3x meals
🎯 PER MEAL     : ~733 kcal | 60.0g Prot
--------------------------------------------------
⏰ MEAL #1
   Menu    : Gun powder chutney
   Portion : 2.5 x Serving (312 kcal/srv)
   Total   : 781 kcal | 53.9g Prot
------------------------------
⏰ MEAL #2
   Menu    : Lobster Roll Sandwich
   Portion : 1.5 x Serving (450 kcal/srv)
   Total   : 675 kcal | 30.0g Prot
------------------------------
⏰ MEAL #3
   Menu    : Maa chaane ki dal
   Portion : 2.0 x Serving (345 kcal/srv)
   Total   : 689 kcal | 39.6g Prot
------------------------------
📊 DAILY SUMMARY
   Calories : 2145 / 2200 (98%)
   Protein  : 123g / 180g
